# Chapter 23: Capstone (Reference)

## Learning Objectives

- Compose the full airlock for two contrasting PRs, reusing Chapter 17's decide()
- Build a durable, JSON-lines audit trail from each Decision
- Answer "why did PR #N merge?" with one function call, fulfilling Chapter 01 §9
- Confirm the audit trail generalizes to any number of gates

## Setup

The next cell sets up reproducibility and the `PRA_MODE` toggle, and clears any prior audit log from earlier runs so this notebook's output is deterministic. You should see `PRA_MODE = 'fixture'` printed by default.

In [1]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run against a real repo"

print(f"PRA_MODE = {PRA_MODE!r}")

AUDIT_LOG_PATH = (
    REPO_ROOT / os.environ.get("PRA_OUTPUT_DIR", "output") / "audit_log.jsonl"
)
AUDIT_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
AUDIT_LOG_PATH.write_text("")  # start each run from a clean, empty log

PRA_MODE = 'fixture'


0

## 1. Compose and Log Two Contrasting PRs

The next cell runs the full airlock for a trivial safe fix (PR #301) and a large migration (PR #302), logging both. You should see #301 MERGE with all three gates PASS, and #302 HOLD on Gate 3's hard-ceiling rationale.

In [2]:
from labs.lab_17_airlock import decide
from labs.lab_23_capstone import build_audit_entry, append_audit_log
from pr_automerge.models import PRMetadata
from pr_automerge.render import gate_table, decision_line

safe_pr = PRMetadata(
    number=301,
    title="fix: correct typo in sandbox README",
    base="main",
    head="fix/readme-typo",
    additions=2,
    deletions=1,
    changed_files=1,
    critical_path_hits=0,
)
risky_pr = PRMetadata(
    number=302,
    title="migrate: restructure sandbox data layer",
    base="main",
    head="migrate/sandbox-data-layer",
    additions=700,
    deletions=200,
    changed_files=30,
    critical_path_hits=2,
)

for pr in (safe_pr, risky_pr):
    decision = decide(
        pr,
        main_exists=True,
        protection_configured=True,
        required_checks_registered=True,
        auto_merge_enabled=True,
        ci_conclusion="success",
    )
    gate_table(decision.gates)
    decision_line(decision)
    append_audit_log(build_audit_entry(decision), AUDIT_LOG_PATH)

GATE                    STATUS  RATIONALE
-----------------------------------------
gate1_repo_readiness    PASS    main exists, protection configured, checks registered, auto-merge enabled
gate2_pr_health         PASS    CI conclusion is 'success'
gate3_risk_scoring      PASS    risk=5.9 <= threshold=70

PR #301: MERGE
GATE                    STATUS  RATIONALE
-----------------------------------------
gate1_repo_readiness    PASS    main exists, protection configured, checks registered, auto-merge enabled
gate2_pr_health         PASS    CI conclusion is 'success'
gate3_risk_scoring      FAIL    900 lines exceeds the hard ceiling of 500 — never auto-merged regardless of score

PR #302: HOLD
  blocked by: gate3_risk_scoring — 900 lines exceeds the hard ceiling of 500 — never auto-merged regardless of score


## 2. The One-Command Answer

The next cell queries the audit log for each PR independently. You should see each PR's full, gate-by-gate rationale -- not just a bare pass/fail.

In [3]:
from labs.lab_23_capstone import explain_decision

print(explain_decision(301, AUDIT_LOG_PATH))
print()
print(explain_decision(302, AUDIT_LOG_PATH))

PR #301: MERGED
  gate1_repo_readiness: PASS — main exists, protection configured, checks registered, auto-merge enabled
  gate2_pr_health: PASS — CI conclusion is 'success'
  gate3_risk_scoring: PASS — risk=5.9 <= threshold=70

PR #302: HELD
  gate1_repo_readiness: PASS — main exists, protection configured, checks registered, auto-merge enabled
  gate2_pr_health: PASS — CI conclusion is 'success'
  gate3_risk_scoring: FAIL — 900 lines exceeds the hard ceiling of 500 — never auto-merged regardless of score


## Takeaways & Next Steps

This notebook's takeaway is Chapter 01 §9's promise, kept: every gate's rationale, durably recorded, answerable with one function call -- the last piece of this curriculum's own design, assembled here in the final chapter.

In [4]:
print(
    "You've completed the full 23-chapter curriculum. See Chapter 23 Section 18 for what's next."
)

You've completed the full 23-chapter curriculum. See Chapter 23 Section 18 for what's next.


---

📖 **Reading companion:** [Chapter 23: Capstone](../learning_modules/chapter_23_capstone.md)
🔬 **Try it live:** this chapter's lab is fully offline (no `gh` calls), so `PRA_MODE=live` changes nothing here — nothing to re-run against a real repo.
